# 02. Clinical age model

Train **TabPFN-3.5** as a regressor and compare it with conventional baselines.

Target: `RIDAGEYR`

The final model is evaluated on the held-out NHANES test set and saved as a small account-specific TabPFN model reference.

In [1]:
from pathlib import Path
import sys
import json
import time
import numpy as np
import pandas as pd

root = Path.cwd().resolve()
for candidate in [root, *root.parents]:
    if (candidate / "src").exists() and (candidate / "data").exists():
        root = candidate
        break
sys.path.insert(0, str(root))

from src.config import PROCESSED_DIR, MODELS_DIR, RESULTS_DIR, TARGET_AGE, RANDOM_STATE
from src.data_utils import (load_json, apply_categorical_dtypes)
from src.evaluation import regression_metrics
from src.model_utils import make_age_model, save_model

train = pd.read_csv(PROCESSED_DIR / "train_age.csv")
test = pd.read_csv(PROCESSED_DIR / "test_age.csv")
metadata = load_json(PROCESSED_DIR / "metadata.json")
features = metadata["age_features"]

X_train = train[features].copy()
y_train = train[TARGET_AGE].copy()
X_test = test[features].copy()
y_test = test[TARGET_AGE].copy()

print("Train:", X_train.shape, "Test:", X_test.shape)

Train: (8225, 71) Test: (2057, 71)


In [ ]:
# Categorical features are only converted for TabPFN.
# Conventional baselines keep their original numeric representation.
categorical_features = metadata["categorical_age_features"]

X_train_tabpfn = apply_categorical_dtypes(
    X_train,
    categorical_features)

X_test_tabpfn = apply_categorical_dtypes(
    X_test,
    categorical_features)

print("Categorical features used by TabPFN-3.5:", categorical_features)
print(X_train_tabpfn[categorical_features].dtypes)

Categorical features used by TabPFN-3.5: ['RIAGENDR', 'RIDRETH1', 'DMDEDUC2', 'DMDMARTL', 'SMQ040', 'PAQ180']
RIAGENDR    category
RIDRETH1    category
DMDEDUC2    category
DMDMARTL    category
SMQ040      category
PAQ180      category
dtype: object


In [4]:
# Conventional baselines. TabPFN-3.5 is compared against these; do not report only TabPFN.
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import HistGradientBoostingRegressor, RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline

baselines = {
    "DummyMedian": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("model", DummyRegressor(strategy="median")),
    ]),
    "RandomForest": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("model", RandomForestRegressor(n_estimators=300, random_state=RANDOM_STATE, n_jobs=-1)),
    ]),
    "HistGradientBoosting": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("model", HistGradientBoostingRegressor(random_state=RANDOM_STATE, max_iter=300)),
    ]),
}

rows = []
for name, model in baselines.items():
    t0 = time.time()
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    metrics = regression_metrics(y_test, pred)
    metrics.update({"Model": name, "Seconds": time.time() - t0})
    rows.append(metrics)

baseline_results = pd.DataFrame(rows).set_index("Model")
baseline_results

,MAE,RMSE,R2,Seconds
Model,,,,
DummyMedian,16.861449,19.613846,-0.011339,0.091348
RandomForest,8.101090,10.480258,0.711255,7.614372
HistGradientBoosting,7.139698,9.269626,0.774111,1.545226


In [7]:
# TabPFN-3.5 regression.
# v3.5_default is an explicit model selector in tabpfn-client.
# thinking_effort="high" is the final showcase configuration; use "medium" while debugging.

age_model = make_age_model(thinking_effort="high")
t0 = time.time()
age_model.fit(X_train_tabpfn, y_train)
age_pred = age_model.predict(X_test_tabpfn)
age_seconds = time.time() - t0

tabpfn_age_metrics = regression_metrics(y_test, age_pred)
tabpfn_age_metrics.update({"Model": "TabPFN-3.5", "Seconds": age_seconds})

print(tabpfn_age_metrics)

{'MAE': 5.980290662796102, 'RMSE': 8.06795054057155, 'R2': 0.8288812629613609, 'Model': 'TabPFN-3.5', 'Seconds': 106.00562310218811}


In [ ]:
# TabPFN-3.5 regression.
# v3.5_default is an explicit model selector in tabpfn-client.
# thinking_effort="high" is the final showcase configuration; use "medium" while debugging.

age_model = make_age_model(thinking_effort="high")
t0 = time.time()
age_model.fit(X_train_tabpfn, y_train)
age_pred = age_model.predict(X_test_tabpfn)
age_seconds = time.time() - t0

tabpfn_age_metrics = regression_metrics(y_test, age_pred)
tabpfn_age_metrics.update({"Model": "TabPFN-3.5", "Seconds": age_seconds})

print(tabpfn_age_metrics)

{'MAE': 5.976378184472929, 'RMSE': 8.060830359346388, 'R2': 0.8291831633706326, 'Model': 'TabPFN-3.5', 'Seconds': 140.59975814819336}


In [8]:
# Compare results and save predictions.
all_results = pd.concat([baseline_results.reset_index(), pd.DataFrame([tabpfn_age_metrics])], ignore_index=True)
all_results = all_results[["Model", "MAE", "RMSE", "R2", "Seconds"]]
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
all_results.to_csv(RESULTS_DIR / "age_model_comparison.csv", index=False)

pred_df = pd.DataFrame({
    "ROW_ID": test["ROW_ID"],
    "RIDAGEYR": y_test,
    "AGE_PRED_TABPFN": age_pred,
    "AGE_ACCELERATION": age_pred - y_test,
    "MORTSTAT": test["MORTSTAT"],
})
pred_df.to_csv(RESULTS_DIR / "age_test_predictions.csv", index=False)

MODELS_DIR.mkdir(parents=True, exist_ok=True)
model_record = save_model(age_model, MODELS_DIR / "age_model.json")
with open(MODELS_DIR / "age_model_record.json", "w", encoding="utf-8") as fh:
    json.dump(model_record, fh, indent=2)

print(all_results)

                  Model        MAE       RMSE        R2     Seconds
0           DummyMedian  16.861449  19.613846 -0.011339    0.091348
1          RandomForest   8.101090  10.480258  0.711255    7.614372
2  HistGradientBoosting   7.139698   9.269626  0.774111    1.545226
3            TabPFN-3.5   5.980291   8.067951  0.828881  106.005623


In [8]:
# Compare results and save predictions.
all_results = pd.concat([baseline_results.reset_index(), pd.DataFrame([tabpfn_age_metrics])], ignore_index=True)
all_results = all_results[["Model", "MAE", "RMSE", "R2", "Seconds"]]
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
all_results.to_csv(RESULTS_DIR / "age_model_comparison.csv", index=False)

pred_df = pd.DataFrame({
    "ROW_ID": test["ROW_ID"],
    "RIDAGEYR": y_test,
    "AGE_PRED_TABPFN": age_pred,
    "AGE_ACCELERATION": age_pred - y_test,
    "MORTSTAT": test["MORTSTAT"],
})
pred_df.to_csv(RESULTS_DIR / "age_test_predictions.csv", index=False)

MODELS_DIR.mkdir(parents=True, exist_ok=True)
model_record = save_model(age_model, MODELS_DIR / "age_model.json")
with open(MODELS_DIR / "age_model_record.json", "w", encoding="utf-8") as fh:
    json.dump(model_record, fh, indent=2)

print(all_results)

                  Model        MAE       RMSE        R2     Seconds
0           DummyMedian  16.861449  19.613846 -0.011339    0.099478
1          RandomForest   8.101090  10.480258  0.711255    7.713726
2  HistGradientBoosting   7.139698   9.269626  0.774111    1.360397
3            TabPFN-3.5   5.976378   8.060830  0.829183  140.599758


In [9]:
# Optional: inspect the error by age band.
pred_df["AGE_BAND"] = pd.cut(
    pred_df["RIDAGEYR"],
    bins=[19, 39, 59, 79, 85, np.inf],
    labels=["20–39", "40–59", "60–79", "80–84", "85+"],
    right=True,
)
print(pred_df.groupby("AGE_BAND", observed=False)["AGE_ACCELERATION"].agg(["count", "mean", "median", "std"]))

          count      mean    median       std
AGE_BAND                                     
20–39       738  4.813446  3.642312  7.039711
40–59       583 -0.495770 -0.527721  7.221740
60–79       550 -2.669355 -1.639191  7.670964
80–84       186 -6.327601 -4.882683  5.743655
85+           0       NaN       NaN       NaN
